In [ ]:
import pandas as pd

df = pd.read_csv("data/raw.csv")

C:\Users\silly\OneDrive\Personal Projects\Sales Forecasting Project


In [2]:
###########Date Features############
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Quarter"] = df["Date"].dt.quarter
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["Season"] = df["Month"].apply(get_season)
df = pd.get_dummies(df, columns=["Season"])

In [3]:
##########Lag Features##################
df = df.sort_values(["Store", "Date"])

df["Sales_Lag_1"] = df.groupby("Store")["Weekly_Sales"].shift(1)

df["Sales_Lag_4"] = df.groupby("Store")["Weekly_Sales"].shift(4)


In [4]:
###### Rolling Averages ##############
df["Rolling_Mean_4"] = (
    df.groupby("Store")["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(4).mean())
)

df["Rolling_Mean_12"] = (
    df.groupby("Store")["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(12).mean())
)

df["Rolling_STD_4"] = (
    df.groupby("Store")["Weekly_Sales"]
      .transform(lambda x: x.shift(1).rolling(4).std())
)

In [ ]:
#####Check Missing values#############
print(df.isnull().sum())

df = df.dropna()

print(df.head())

df.to_csv("data/walmart_sales_engineered.csv", index=False)

Store                0
Date                 0
Weekly_Sales         0
Holiday_Flag         0
Temperature          0
Fuel_Price           0
CPI                  0
Unemployment         0
Year                 0
Month                0
Quarter              0
Week                 0
DayOfWeek            0
Season_Fall          0
Season_Spring        0
Season_Summer        0
Season_Winter        0
Sales_Lag_1         45
Sales_Lag_4        180
Rolling_Mean_4     180
Rolling_Mean_12    540
Rolling_STD_4      180
dtype: int64
    Store       Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
12      1 2010-04-30    1425100.71             0        67.41       2.780   
13      1 2010-05-07    1603955.12             0        72.55       2.835   
14      1 2010-05-14    1494251.50             0        74.78       2.854   
15      1 2010-05-21    1399662.07             0        76.44       2.826   
16      1 2010-05-28    1432069.95             0        80.44       2.759   

           CPI  Un